# 07 — BQ5: Le 3 principali raccomandazioni operative per la retention degli studenti

> **Notebook 07 di 7** | Learning Retention Analytics  
> Quinta business question: sintesi dei risultati BQ1–BQ4 in raccomandazioni concrete e prioritizzate per un operatore di piattaforma.

## Scopo e ambito

Questo notebook risponde a **BQ5: Quali sono le 3 principali raccomandazioni operative per un operatore di piattaforma?** — l'ultima business question e il punto culminante di questo progetto.

I notebook 03–06 hanno stabilito *cosa accade* (BQ1: tempistica del dropout), *cosa lo predice* (BQ2: segnali precoci), *se contano di più i dati demografici o il comportamento* (BQ3: vince il comportamento) e *come differiscono i corsi* (BQ4: design del corso vs retention). Questo notebook **trasforma quei risultati in azione**.

**Cosa copre questo notebook:**
- Dimensionamento dei tre segmenti target identificati attraverso BQ1–BQ4
- Quantificazione della sovrapposizione tra segmenti (gli studenti possono appartenere a più segmenti)
- Costruzione di tre schede di raccomandazione con: segmento, intervento, impatto atteso, costo
- Classificazione degli interventi in una matrice di priorità (impatto vs costo)
- Proposta di una roadmap di implementazione per fasi

**Cosa questo notebook NON fa:**
- Nessun nuovo test statistico — tutte le evidenze provengono da BQ1–BQ4
- Nessuna affermazione causale — le stime di impatto sono proiezioni, non effetti misurati
- Nessun disegno di A/B test — questo è il passo successivo naturale dopo il deployment degli interventi

**Cosa è venuto prima:**
- **NB03** (BQ1): dove e quando gli studenti abbandonano — curve di dropout, rilevamento dei cliff
- **NB04** (BQ2): segnali comportamentali precoci che predicono il dropout — effect size, dose-response
- **NB05** (BQ3): dati demografici vs comportamento — il comportamento predice l'esito 2–5× più fortemente
- **NB06** (BQ4): design del corso vs retention — profili descrittivi dei corsi, correlazioni esplorative

> **Trasferibilità metodologica:** Questo pattern di sintesi — dimensionamento dei segmenti → design dell'intervento → stima dell'impatto → prioritizzazione — è il "playbook standard per gli interventi sul churn" nella product analytics SaaS. I tre segmenti (utenti ghost, non-adottanti di funzionalità, disimpegnati precoci) si mappano direttamente su churn da abbonamento, retention nelle app fitness e conversione freemium.

## Indice

1. [Configurazione ambiente](#1.-Configurazione-ambiente)
2. [Dimensionamento dei segmenti — Le tre popolazioni target](#2.-Dimensionamento-dei-segmenti-—-Le-tre-popolazioni-target)
3. [Sovrapposizione dei segmenti](#3.-Sovrapposizione-dei-segmenti)
4. [Raccomandazione 1 — Attivazione degli studenti ghost](#4.-Raccomandazione-1-—-Attivazione-degli-studenti-ghost)
5. [Raccomandazione 2 — Checkpoint della prima valutazione](#5.-Raccomandazione-2-—-Checkpoint-della-prima-valutazione)
6. [Raccomandazione 3 — Campagna di re-engagement alla settimana 3](#6.-Raccomandazione-3-—-Campagna-di-re-engagement-alla-settimana-3)
7. [Matrice di priorità](#7.-Matrice-di-priorità)
8. [Roadmap di implementazione](#8.-Roadmap-di-implementazione)
9. [Limitazioni e avvertenze](#9.-Limitazioni-e-avvertenze)
10. [Conclusioni chiave](#10.-Conclusioni-chiave)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Configurazione ambiente

Configuriamo gli import, i parametri di visualizzazione e le funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- Tutte le query al database passano attraverso `src.db.connection.execute_query()` — il livello di astrazione DB del progetto (ADR-003).
- La query SQL principale di BQ5 risiede in `sql/queries/q_bq5_segment_sizing.sql` e viene caricata a runtime dal disco. Dimensiona tre segmenti di intervento da `v_student_enriched` e `v_engagement_early`.
- Query SQL inline aggiuntive calcolano le stime di impatto per ciascuna raccomandazione. Sono specifiche per la narrativa di sintesi di questo notebook e non riutilizzabili come query autonome (coerente con il pattern di query inline in NB03).
- Nessun import di test statistici — questo è un notebook di sintesi, tutte le evidenze provengono da NB03–NB06.
- Le figure vengono salvate in `reports/figures/` a 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

# Segment palette: one color per intervention segment for consistent
# identification across all figures. Colors chosen for semantic clarity:
# red = most critical (ghost), orange = medium (assessment), blue = re-engagement.
PALETTE_SEGMENT = {
    'Ghost students': '#C44E52',
    'Assessment non-submitters': '#DD8452',
    'Early disengagers': '#4C72B0',
}
SEGMENT_ORDER = list(PALETTE_SEGMENT.keys())

# Shared axis labels and section headers — defined as constants to avoid
# duplicated string literals flagged by static analysis
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NON_COMPLETION_RATE = 'Non-completion rate (%)'
LABEL_NUM_STUDENTS = 'Number of students'
HEADER_SCENARIO = '=== Scenario Analysis ==='

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ5 query from SQL file ---
_bq5_sql_path = QUERIES_DIR / 'q_bq5_segment_sizing.sql'
BQ5_SQL = _bq5_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ5 query from: {_bq5_sql_path.name} ({len(BQ5_SQL):,} chars)')

# --- Prerequisite check ---
# BQ5 query depends on v_student_enriched (main student table) and
# v_engagement_early (early behavioral metrics for segment classification).
try:
    _check_se = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_ee = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_se = _check_se['n'].iloc[0]
    _n_ee = _check_ee['n'].iloc[0]
    if _n_se == 0 or _n_ee == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_se:>12,} rows')
    print(f'  v_engagement_early:  {_n_ee:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Dimensionamento dei segmenti — Le tre popolazioni target

La query BQ5 (`q_bq5_segment_sizing.sql`) dimensiona tre segmenti di studenti definiti da **criteri osservabili e azionabili** — non demografici. Ogni segmento rappresenta un gruppo in cui un intervento mirato potrebbe ridurre il non-completamento:

| Segmento | Definizione | Razionale |
|----------|------------|-----------|
| **Studenti ghost** | ≤1 giorno attivo E <10 click nei primi 28 giorni | Iscritti ma mai iniziato in modo significativo. BQ2 (NB04) ha mostrato che zero engagement precoce è il predittore più forte di dropout. |
| **Non-submitter delle valutazioni** | Nessuna valutazione consegnata nei primi 28 giorni | Mancare la prima scadenza è un potente segnale binario. BQ2 ha identificato `submitted_first_assessment` come predittore chiave. |
| **Disimpegnati precoci** | Attività VLE nei giorni 0–14 ma zero attività nei giorni 15–28 | Hanno iniziato ma perso il momentum. BQ1 (NB03) ha mostrato cliff di dropout a metà corso, spesso in corrispondenza dei punti di valutazione. |

**Principio di design:** Tutti e tre i segmenti sono definiti dal *comportamento*, non dai dati demografici. Questo è coerente con il risultato di BQ3 (NB05) che i segnali comportamentali predicono l'esito 2–5× più fortemente delle feature demografiche. Gli interventi che mirano al comportamento sono sia più efficaci sia più etici.

In [ ]:
# --- Load BQ5 segment sizing data ---
df_bq5 = execute_query(BQ5_SQL)

total_students = int(df_bq5['total_students'].iloc[0])

# Build a structured DataFrame for the three segments
# so we can plot and reference them consistently across sections
segments = pd.DataFrame({
    'segment': SEGMENT_ORDER,
    'n': [
        int(df_bq5['n_ghost'].iloc[0]),
        int(df_bq5['n_non_submitter'].iloc[0]),
        int(df_bq5['n_early_disengager'].iloc[0]),
    ],
    'pct_of_total': [
        float(df_bq5['pct_ghost'].iloc[0]),
        float(df_bq5['pct_non_submitter'].iloc[0]),
        float(df_bq5['pct_early_disengager'].iloc[0]),
    ],
    'non_completion_rate': [
        float(df_bq5['ghost_non_completion_rate_pct'].iloc[0]),
        float(df_bq5['non_submitter_non_completion_rate_pct'].iloc[0]),
        float(df_bq5['disengager_non_completion_rate_pct'].iloc[0]),
    ],
})

# Overall non-completion rate for context (baseline)
overall_non_completion = execute_query('''
    SELECT ROUND(100.0 * SUM(CASE WHEN completed = 0 THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS rate
    FROM v_student_enriched
''')['rate'].iloc[0]

print(f'BQ5 segment sizing: {total_students:,} total enrollments')
print(f'Overall non-completion rate: {overall_non_completion}%')
print()
print('=== Segment Summary ===')
print(segments.to_string(index=False))
print()
# How much worse each segment is compared to the overall rate
for _, row in segments.iterrows():
    excess = row['non_completion_rate'] - overall_non_completion
    print(f"  {row['segment']}: {row['non_completion_rate']:.1f}% non-completion "
          f"(+{excess:.1f} pp vs overall)")

In [ ]:
# --- Figure: Segment sizing overview ---
# 1x2 panel: segment sizes (left) and non-completion rates (right).
# Segments ordered by severity (SEGMENT_ORDER) for consistent reading.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

y_pos = np.arange(len(segments))
colors = [PALETTE_SEGMENT[s] for s in segments['segment']]

# --- Left: segment size ---
ax1.barh(y_pos, segments['n'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(segments.iterrows()):
    ax1.text(
        row['n'] + total_students * 0.01, i,
        f"{row['n']:,}  ({row['pct_of_total']:.1f}%)",
        va='center', fontsize=10, color='#333333',
    )
ax1.set_yticks(y_pos)
ax1.set_yticklabels(segments['segment'])
ax1.set_xlabel(LABEL_NUM_STUDENTS)
ax1.set_title('Segment Size')
sns.despine(ax=ax1)

# --- Right: non-completion rate ---
ax2.barh(y_pos, segments['non_completion_rate'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(segments.iterrows()):
    ax2.text(
        row['non_completion_rate'] + 1, i,
        f"{row['non_completion_rate']:.1f}%",
        va='center', fontsize=10, color='#333333',
    )
# Overall baseline reference line
ax2.axvline(
    x=overall_non_completion, color='gray', linestyle='--', linewidth=1,
    label=f'Overall: {overall_non_completion:.1f}%',
)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(segments['segment'])
ax2.set_xlabel(LABEL_NON_COMPLETION_RATE)
ax2.set_title('Non-completion Rate by Segment')
ax2.set_xlim(0, 105)
ax2.legend(loc='lower right')
sns.despine(ax=ax2)

fig.suptitle(
    'BQ5 Segment Sizing — Who Should We Target?\n'
    f'(total enrollments: {total_students:,})',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '07_segment_sizing_overview')
plt.show()

> **Profilo dei segmenti:** Tutti e tre i segmenti mostrano tassi di non-completamento sostanzialmente superiori alla baseline complessiva. Il gap tra il tasso di ciascun segmento e la media della piattaforma quantifica il "non-completamento in eccesso" — la porzione potenzialmente affrontabile attraverso un intervento mirato.
>
> **Nota:** Questi segmenti non sono mutuamente esclusivi. Uno studente ghost che non ha mai acceduto al VLE quasi certamente non ha nemmeno consegnato una valutazione. La sezione successiva quantifica questa sovrapposizione per evitare il doppio conteggio nella stima dell'impatto aggregato.

## 3. Sovrapposizione dei segmenti

Gli studenti possono appartenere a più segmenti contemporaneamente. Comprendere la sovrapposizione è fondamentale per due ragioni:
1. **Stima dell'impatto:** Se la maggior parte degli studenti ghost sono anche non-submitter, i due interventi mirano in gran parte alle stesse persone — il loro impatto non dovrebbe essere sommato in modo ingenuo.
2. **Sequenziamento degli interventi:** Uno studente in più segmenti riceverebbe più interventi; dobbiamo stabilire una priorità su quale riceve per primo.

La query seguente riproduce la classificazione dei segmenti BQ5 a livello di studente (invece di aggregare in una singola riga) e conta le intersezioni a coppie e a tre.

In [ ]:
# --- Segment overlap analysis ---
# Inline query: reproduce the BQ5 segment flags at the student level
# and count pairwise/three-way intersections.
# This mirrors the CTE logic in q_bq5_segment_sizing.sql but selects
# per-student flags instead of aggregating to a single row.
OVERLAP_SQL = '''
WITH early_activity AS (
    SELECT id_student, code_module, code_presentation,
           SUM(sum_click) AS total_clicks_0_14
    FROM studentVle
    WHERE date BETWEEN 0 AND 14
    GROUP BY id_student, code_module, code_presentation
),
late_activity AS (
    SELECT id_student, code_module, code_presentation,
           SUM(sum_click) AS total_clicks_15_28
    FROM studentVle
    WHERE date BETWEEN 15 AND 28
    GROUP BY id_student, code_module, code_presentation
),
early_assessments AS (
    SELECT DISTINCT sa.id_student, a.code_module, a.code_presentation
    FROM studentAssessment sa
    JOIN assessments a ON sa.id_assessment = a.id_assessment
    WHERE a.date <= 28
),
student_segments AS (
    SELECT
        se.id_student, se.code_module, se.code_presentation,
        CASE WHEN COALESCE(ee.active_days_first_28, 0) <= 1
                  AND COALESCE(ee.total_clicks_first_28, 0) < 10
             THEN 1 ELSE 0 END AS is_ghost,
        CASE WHEN ea.id_student IS NULL
             THEN 1 ELSE 0 END AS is_non_submitter,
        CASE WHEN early_act.total_clicks_0_14 IS NOT NULL
                  AND late_act.total_clicks_15_28 IS NULL
             THEN 1 ELSE 0 END AS is_early_disengager
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    LEFT JOIN early_assessments ea
        ON se.id_student = ea.id_student
        AND se.code_module = ea.code_module
        AND se.code_presentation = ea.code_presentation
    LEFT JOIN early_activity early_act
        ON se.id_student = early_act.id_student
        AND se.code_module = early_act.code_module
        AND se.code_presentation = early_act.code_presentation
    LEFT JOIN late_activity late_act
        ON se.id_student = late_act.id_student
        AND se.code_module = late_act.code_module
        AND se.code_presentation = late_act.code_presentation
)
SELECT
    COUNT(*) AS total,
    SUM(is_ghost) AS n_ghost,
    SUM(is_non_submitter) AS n_non_sub,
    SUM(is_early_disengager) AS n_disengager,
    -- Pairwise overlaps
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 1
             THEN 1 ELSE 0 END) AS ghost_and_nonsub,
    SUM(CASE WHEN is_ghost = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS ghost_and_disengager,
    SUM(CASE WHEN is_non_submitter = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS nonsub_and_disengager,
    -- Three-way overlap
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS all_three,
    -- Exclusive membership (belongs to exactly this one segment)
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 0 AND is_early_disengager = 0
             THEN 1 ELSE 0 END) AS ghost_only,
    SUM(CASE WHEN is_ghost = 0 AND is_non_submitter = 1 AND is_early_disengager = 0
             THEN 1 ELSE 0 END) AS nonsub_only,
    SUM(CASE WHEN is_ghost = 0 AND is_non_submitter = 0 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS disengager_only,
    -- Union: in at least one segment
    SUM(CASE WHEN is_ghost = 1 OR is_non_submitter = 1 OR is_early_disengager = 1
             THEN 1 ELSE 0 END) AS in_any_segment
FROM student_segments
'''

df_overlap = execute_query(OVERLAP_SQL)
row = df_overlap.iloc[0]

# --- Print overlap summary ---
print('=== Segment Overlap Analysis ===')
print(f"Total enrollments:       {int(row['total']):>8,}")
print(f"In at least one segment: {int(row['in_any_segment']):>8,} "
      f"({100.0 * row['in_any_segment'] / row['total']:.1f}%)")
print()
print('Pairwise overlaps:')
print(f"  Ghost ∩ Non-submitter:     {int(row['ghost_and_nonsub']):>6,}")
print(f"  Ghost ∩ Disengager:        {int(row['ghost_and_disengager']):>6,}")
print(f"  Non-submitter ∩ Disengager:{int(row['nonsub_and_disengager']):>6,}")
print(f"  All three:                 {int(row['all_three']):>6,}")
print()
print('Exclusive membership (this segment only):')
print(f"  Ghost only:                {int(row['ghost_only']):>6,}")
print(f"  Non-submitter only:        {int(row['nonsub_only']):>6,}")
print(f"  Disengager only:           {int(row['disengager_only']):>6,}")

In [ ]:
# --- Figure: Segment overlap ---
# Horizontal bar showing exclusive and overlapping membership counts.
# This helps understand how much the interventions would target the same students.
overlap_data = pd.DataFrame({
    'category': [
        'Ghost only',
        'Non-submitter only',
        'Disengager only',
        'Ghost + Non-submitter',
        'Non-submitter + Disengager',
        'Ghost + Disengager',
        'All three',
    ],
    'n': [
        int(row['ghost_only']),
        int(row['nonsub_only']),
        int(row['disengager_only']),
        # Pairwise exclusive: subtract three-way overlap to avoid double-counting
        int(row['ghost_and_nonsub']) - int(row['all_three']),
        int(row['nonsub_and_disengager']) - int(row['all_three']),
        int(row['ghost_and_disengager']) - int(row['all_three']),
        int(row['all_three']),
    ],
})

# Filter out zero-count categories for cleaner visualization
overlap_data = overlap_data[overlap_data['n'] > 0].sort_values('n', ascending=True)

# Color mapping: exclusive categories get the segment color,
# overlap categories get a neutral gray
overlap_colors = []
for cat in overlap_data['category']:
    if cat == 'Ghost only':
        overlap_colors.append(PALETTE_SEGMENT['Ghost students'])
    elif cat == 'Non-submitter only':
        overlap_colors.append(PALETTE_SEGMENT['Assessment non-submitters'])
    elif cat == 'Disengager only':
        overlap_colors.append(PALETTE_SEGMENT['Early disengagers'])
    else:
        overlap_colors.append('#999999')

fig, ax = plt.subplots(figsize=FIG_SIZE)
y_pos = np.arange(len(overlap_data))

ax.barh(y_pos, overlap_data['n'].values, color=overlap_colors, edgecolor='white')
for i, (_, cat_row) in enumerate(overlap_data.iterrows()):
    n_val = cat_row['n']
    pct = 100.0 * n_val / total_students
    ax.text(
        n_val + total_students * 0.005, i,
        f'{n_val:,}  ({pct:.1f}%)',
        va='center', fontsize=9, color='#333333',
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(overlap_data['category'].values)
ax.set_xlabel(LABEL_NUM_STUDENTS)
ax.set_title(
    'Segment Overlap — Exclusive and Shared Membership\n'
    '(gray bars = students in multiple segments)'
)
sns.despine()
fig.tight_layout()
save_fig(fig, '07_segment_overlap')
plt.show()

# --- Key overlap metric for downstream impact estimation ---
# Percentage of ghost students who are also non-submitters
ghost_total = int(row['n_ghost'])
ghost_nonsub_overlap = int(row['ghost_and_nonsub'])
if ghost_total > 0:
    overlap_pct = 100.0 * ghost_nonsub_overlap / ghost_total
    print(f'\nGhost ∩ Non-submitter overlap: {ghost_nonsub_overlap:,} of '
          f'{ghost_total:,} ghost students ({overlap_pct:.0f}%)')
    print('  → Ghost activation and assessment reminders would largely target '
          'the same students.')

> **Insight sulla sovrapposizione:** Studenti ghost e non-submitter delle valutazioni si sovrappongono pesantemente — uno studente che non accede mai al VLE non può consegnare una valutazione. Questo significa che le **Raccomandazioni 1 e 2 mirano in gran parte alla stessa popolazione** da angolazioni diverse. I disimpegnati precoci, per definizione, hanno avuto *qualche* attività iniziale, quindi si sovrappongono meno con gli studenti ghost. Questo rende la Raccomandazione 3 un intervento indipendente che mira a una diversa modalità di fallimento.
>
> **Implicazione per la stima dell'impatto:** A causa della sovrapposizione ghost–non-submitter, l'impatto combinato di tutti e tre gli interventi è **inferiore alla somma degli impatti individuali**. Ogni sezione di raccomandazione sotto stima l'impatto indipendentemente; la Matrice di Priorità (Sezione 7) presenta quelle stime indipendenti e include la sovrapposizione come considerazione interpretativa nella lettura dei totali combinati.

## 4. Raccomandazione 1 — Attivazione degli studenti ghost

### Il problema

Gli studenti ghost si iscrivono ma non interagiscono mai in modo significativo con il corso. Rappresentano la forma più grave di non-completamento: non un fallimento nel persistere, ma un fallimento nell'*iniziare*.

### Evidenze da BQ1–BQ4

- **BQ2 (NB04):** L'engagement precoce (giorni attivi, click totali nei primi 28 giorni) ha il più grande effect size tra tutti i predittori comportamentali. Gli studenti con zero attività precoce hanno tassi di completamento prossimi allo zero.
- **BQ3 (NB05):** Il comportamento predice l'esito 2–5× più fortemente dei dati demografici. Questo significa che dobbiamo mirare a *cosa fanno gli studenti* (o non fanno), non a *chi sono*.
- **BQ1 (NB03):** Una frazione significativa dei ritiri avviene nelle prime due settimane — prima che la maggior parte degli studenti abbia stabilito una routine di studio.

### Intervento proposto

| Elemento | Dettagli |
|----------|----------|
| **Trigger** | Lo studente ha zero attività VLE entro il giorno 3 del corso |
| **Azione** | Sequenza di attivazione automatizzata: email di benvenuto al giorno 3 con link "primo passo" a una singola risorsa facile, follow-up al giorno 7 se ancora inattivo |
| **Canale** | Email + notifica in-platform |
| **Costo** | **Basso** — solo automazione email, nessuna modifica alla piattaforma richiesta |

### Stima dell'impatto

In [ ]:
# --- Impact estimation: Ghost student activation ---
# Compare completion rates between ghost students and active students
# to quantify the gap that activation could partially close.
df_ghost_impact = execute_query('''
    SELECT
        CASE
            WHEN COALESCE(ee.active_days_first_28, 0) <= 1
                 AND COALESCE(ee.total_clicks_first_28, 0) < 10
            THEN 'Ghost'
            ELSE 'Active'
        END AS student_type,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY 1
''')

ghost_row = df_ghost_impact[df_ghost_impact['student_type'] == 'Ghost'].iloc[0]
active_row = df_ghost_impact[df_ghost_impact['student_type'] == 'Active'].iloc[0]

ghost_n = int(ghost_row['n'])
ghost_rate = float(ghost_row['completion_rate_pct'])
active_rate = float(active_row['completion_rate_pct'])

print('=== Ghost vs Active Completion Rates ===')
print(df_ghost_impact.to_string(index=False))
print()

# Scenario analysis: if activation emails convert X% of ghost students
# to minimal engagement, and those converted students achieve the
# platform-average completion rate (conservative estimate)

print(HEADER_SCENARIO)
print(f'Ghost students: {ghost_n:,} with {ghost_rate:.1f}% completion rate')
print(f'Active students: {active_rate:.1f}% completion rate')
print(f'Platform average: {100.0 - overall_non_completion:.1f}% completion rate')
print()

# Store scenario results for the priority matrix
ghost_scenarios = {}
for conversion_pct in [10, 20, 30]:
    converted = int(ghost_n * conversion_pct / 100)
    # Conservative: converted students achieve the overall average, not the active rate
    target_rate = (100.0 - overall_non_completion) / 100.0
    current_rate = ghost_rate / 100.0
    additional_completions = int(converted * (target_rate - current_rate))
    ghost_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% of ghosts activate → ~{additional_completions:,} '
          f'additional completions ({converted:,} converted)')

# Save the middle scenario for the priority matrix
rec1_impact = ghost_scenarios[20]

> **Interpretazione:** Il gap nel tasso di completamento tra studenti ghost e attivi è sostanziale. Anche un tasso di attivazione modesto (20%) produrrebbe completamenti aggiuntivi significativi perché il segmento è grande e il gap è ampio.
>
> **Trasparenza sulle assunzioni:** Lo scenario assume che gli studenti convertiti raggiungano il tasso di completamento *medio della piattaforma* — una stima conservativa. In pratica, gli studenti che si attivano in ritardo potrebbero avere performance peggiori rispetto a quelli che hanno iniziato puntualmente. Questo ridurrebbe l'impatto stimato.

## 5. Raccomandazione 2 — Checkpoint della prima valutazione

### Il problema

Gli studenti che mancano la scadenza della prima valutazione stanno segnalando il disimpegno. Mancare questo milestone precoce interrompe il ciclo di feedback che tiene gli studenti connessi al corso — perdono il senso di progresso che la valutazione precoce fornisce.

### Evidenze da BQ1–BQ4

- **BQ2 (NB04):** `submitted_first_assessment` è un potente predittore binario del completamento. L'effect size è tra i più grandi di tutti i segnali precoci.
- **BQ4 (NB06):** I corsi con maggiore densità di valutazioni (checkpoint più frequenti) mostrano pattern suggestivi con la retention. Valutazioni regolari potrebbero fornire una struttura che aiuta gli studenti a restare in carreggiata.
- **BQ1 (NB03):** I cliff di dropout spesso coincidono con le scadenze delle valutazioni, suggerendo che le valutazioni sono punti decisionali dove gli studenti si impegnano o abbandonano.

### Intervento proposto

| Elemento | Dettagli |
|----------|----------|
| **Trigger** | La scadenza della prima valutazione è tra 3 giorni e lo studente non ha consegnato |
| **Azione** | Promemoria automatizzato con anteprima della valutazione: "Ecco cosa aspettarti — la prima valutazione copre X e richiede approssimativamente Y minuti" |
| **Canale** | Email + notifica in-platform + SMS opzionale |
| **Costo** | **Medio** — richiede un sistema di notifiche consapevole delle scadenze integrato con il calendario del corso |

### Stima dell'impatto

In [ ]:
# --- Impact estimation: First assessment checkpoint ---
# Compare completion rates between students who submitted the first
# assessment (due within 28 days) and those who did not.
df_assess_impact = execute_query('''
    SELECT
        CASE
            WHEN ea.id_student IS NOT NULL THEN 'Submitted'
            ELSE 'Not submitted'
        END AS assessment_status,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN (
        SELECT DISTINCT sa.id_student, a.code_module, a.code_presentation
        FROM studentAssessment sa
        JOIN assessments a ON sa.id_assessment = a.id_assessment
        WHERE a.date <= 28
    ) ea
        ON se.id_student = ea.id_student
        AND se.code_module = ea.code_module
        AND se.code_presentation = ea.code_presentation
    GROUP BY 1
''')

sub_row = df_assess_impact[df_assess_impact['assessment_status'] == 'Submitted'].iloc[0]
nosub_row = df_assess_impact[df_assess_impact['assessment_status'] == 'Not submitted'].iloc[0]

nosub_n = int(nosub_row['n'])
nosub_rate = float(nosub_row['completion_rate_pct'])
sub_rate = float(sub_row['completion_rate_pct'])

print('=== Submitter vs Non-submitter Completion Rates ===')
print(df_assess_impact.to_string(index=False))
print()

# Scenario analysis: if reminders convert X% of non-submitters to submitters
print(HEADER_SCENARIO)
print(f'Non-submitters: {nosub_n:,} with {nosub_rate:.1f}% completion rate')
print(f'Submitters: {sub_rate:.1f}% completion rate')
print()

assess_scenarios = {}
for conversion_pct in [10, 15, 25]:
    converted = int(nosub_n * conversion_pct / 100)
    # Converted students achieve submitter rate (they actually submitted)
    additional_completions = int(converted * (sub_rate - nosub_rate) / 100.0)
    assess_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% of non-submitters submit → ~{additional_completions:,} '
          f'additional completions ({converted:,} converted)')

# Save the middle scenario for the priority matrix
rec2_impact = assess_scenarios[15]

> **Interpretazione:** Il gap tra chi consegna e chi non consegna è netto. La consegna della valutazione è sia un *segnale* (rivela l'impegno) sia un *meccanismo* (crea accountability). Questa doppia natura la rende un punto di intervento ideale.
>
> **Cautela sulla causalità:** Consegnare la valutazione potrebbe non *causare* il completamento — entrambi potrebbero essere guidati da un fattore motivazionale sottostante. L'intervento funziona solo se il promemoria spinge studenti che *avrebbero consegnato* con una piccola spinta, non se costringe studenti riluttanti attraverso un cancello. Per questo le assunzioni sul tasso di conversione sono conservative (10–25%).

## 6. Raccomandazione 3 — Campagna di re-engagement alla settimana 3

### Il problema

I disimpegnati precoci sono studenti che hanno *iniziato* il corso — hanno avuto attività VLE nelle prime due settimane — ma poi si sono fermati completamente nelle settimane 3–4. A differenza degli studenti ghost che non hanno mai iniziato, questi studenti hanno dimostrato un interesse iniziale ma hanno perso il momentum.

### Evidenze da BQ1–BQ4

- **BQ1 (NB03):** Le curve di dropout mostrano cliff a metà corso che spesso si allineano con le scadenze delle valutazioni o la transizione dai contenuti introduttivi a quelli core. Le settimane 3–4 sono un punto di inflessione critico.
- **BQ2 (NB04):** `last_active_day_in_window` è un predittore significativo — gli studenti la cui ultima attività è precoce nella finestra hanno bassa probabilità di completare.
- **BQ3 (NB05):** Il segnale comportamentale (calo di attività) è più predittivo di qualsiasi fattore demografico. L'intervento dovrebbe mirare al comportamento, non al profilo dello studente.

### Intervento proposto

| Elemento | Dettagli |
|----------|----------|
| **Trigger** | Lo studente ha avuto ≥1 giorno attivo nei giorni 0–14 ma zero attività nei giorni 15–17 (3 giorni di inattività dopo l'engagement iniziale) |
| **Azione** | Email "Ci manchi" al giorno 18: riepilogo personalizzato dei progressi ("Hai completato il X% delle attività delle settimane 1–2"), social proof ("Studenti come te che si sono ri-impegnati a questo punto hanno completato il corso il Y% delle volte") e un link diretto alla prossima risorsa |
| **Canale** | Email + notifica in-platform |
| **Costo** | **Medio-Alto** — richiede pipeline di tracciamento dell'attività in tempo reale e generazione di messaggi personalizzati |

### Stima dell'impatto

In [ ]:
# --- Impact estimation: Week 3 re-engagement ---
# Compare completion rates between students who sustained activity through
# weeks 3-4 and those who disengaged (had activity in days 0-14 only).
# We restrict to students who had at least SOME initial activity (days 0-14),
# since ghost students are addressed by Recommendation 1.
df_disengage_impact = execute_query('''
    WITH initially_active AS (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM studentVle
        WHERE date BETWEEN 0 AND 14
    ),
    sustained AS (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM studentVle
        WHERE date BETWEEN 15 AND 28
    )
    SELECT
        CASE
            WHEN sus.id_student IS NULL THEN 'Disengaged (weeks 3-4)'
            ELSE 'Sustained activity'
        END AS engagement_pattern,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    JOIN initially_active ia
        ON se.id_student = ia.id_student
        AND se.code_module = ia.code_module
        AND se.code_presentation = ia.code_presentation
    LEFT JOIN sustained sus
        ON se.id_student = sus.id_student
        AND se.code_module = sus.code_module
        AND se.code_presentation = sus.code_presentation
    GROUP BY 1
''')

dis_row = df_disengage_impact[
    df_disengage_impact['engagement_pattern'] == 'Disengaged (weeks 3-4)'
].iloc[0]
sus_row = df_disengage_impact[
    df_disengage_impact['engagement_pattern'] == 'Sustained activity'
].iloc[0]

dis_n = int(dis_row['n'])
dis_rate = float(dis_row['completion_rate_pct'])
sus_rate = float(sus_row['completion_rate_pct'])

print('=== Sustained vs Disengaged Completion Rates ===')
print('(restricted to students with initial activity in days 0-14)')
print(df_disengage_impact.to_string(index=False))
print()

# Scenario analysis
print(HEADER_SCENARIO)
print(f'Early disengagers: {dis_n:,} with {dis_rate:.1f}% completion rate')
print(f'Sustained students: {sus_rate:.1f}% completion rate')
print()

disengage_scenarios = {}
for conversion_pct in [10, 15, 25]:
    converted = int(dis_n * conversion_pct / 100)
    # Re-engaged students achieve a rate between disengaged and sustained
    # (conservative: halfway, not the full sustained rate)
    target_rate = (dis_rate + sus_rate) / 2.0 / 100.0
    current_rate = dis_rate / 100.0
    additional_completions = int(converted * (target_rate - current_rate))
    disengage_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% re-engage → ~{additional_completions:,} '
          f'additional completions ({converted:,} re-engaged)')

# Save the middle scenario for the priority matrix
rec3_impact = disengage_scenarios[15]

> **Interpretazione:** I disimpegnati precoci hanno già dimostrato disponibilità a interagire — non sono studenti ghost. Questo significa che un nudge di re-engagement ha un meccanismo plausibile: ricordare a qualcuno che *era* attivo di tornare. La stima dell'impatto usa un target conservativo (a metà strada tra i tassi dei disimpegnati e di chi ha mantenuto l'attività) perché il re-engagement dopo una pausa è più difficile del momentum sostenuto.
>
> **Perché questo è l'intervento più complesso:** A differenza della semplice automazione email (Racc. 1) o dei promemoria consapevoli delle scadenze (Racc. 2), questo intervento richiede il tracciamento dei pattern di attività individuali in quasi-tempo reale e la generazione di messaggi personalizzati. Il costo di implementazione più alto è giustificato solo se il segmento è sufficientemente grande — il che è confermato dal dimensionamento BQ5.

## 7. Matrice di priorità

La matrice di priorità classifica i tre interventi per **impatto stimato** (completamenti aggiuntivi nello scenario intermedio) vs **costo di implementazione** (Basso / Medio / Medio-Alto). Questo framework aiuta un operatore di piattaforma a decidere *cosa costruire per primo*.

Il costo è valutato su una scala 1–3:
- **1 (Basso):** Solo automazione email, nessuna integrazione con la piattaforma
- **2 (Medio):** Richiede trigger consapevoli delle scadenze o integrazione con il calendario del corso
- **3 (Medio-Alto):** Richiede tracciamento dell'attività in tempo reale e messaggistica personalizzata

In [ ]:
# --- Priority matrix data ---
priority = pd.DataFrame({
    'recommendation': [
        'Ghost Student Activation',
        'First Assessment Checkpoint',
        'Week 3 Re-engagement',
    ],
    'segment': SEGMENT_ORDER,
    'segment_size': [
        segments.loc[0, 'n'],
        segments.loc[1, 'n'],
        segments.loc[2, 'n'],
    ],
    'non_completion_rate': [
        segments.loc[0, 'non_completion_rate'],
        segments.loc[1, 'non_completion_rate'],
        segments.loc[2, 'non_completion_rate'],
    ],
    'est_additional_completions': [rec1_impact, rec2_impact, rec3_impact],
    'cost_score': [1, 2, 3],
    'cost_label': ['Low', 'Medium', 'Medium-High'],
})

# Rank by impact-to-cost ratio so the ordering is data-driven, not hard-coded
priority['impact_per_cost'] = (
    priority['est_additional_completions'] / priority['cost_score']
)
priority = priority.sort_values('impact_per_cost', ascending=False).reset_index(drop=True)
priority['rank'] = priority.index + 1

print('=== Priority Matrix (ranked by impact/cost ratio) ===')
print()
print(priority[[
    'rank', 'recommendation', 'segment_size',
    'non_completion_rate', 'est_additional_completions',
    'cost_label', 'impact_per_cost',
]].to_string(index=False))

print()
total_impact = priority['est_additional_completions'].sum()
print(f'Total estimated additional completions (all 3 interventions): ~{total_impact:,}')
print('''Note: actual total will be lower due to segment overlap
      (ghost ∩ non-submitter).''')

In [ ]:
# --- Figure: Priority matrix ---
# Bubble chart: x = cost (categorical), y = estimated impact,
# bubble size proportional to segment size, colored by segment.
fig, ax = plt.subplots(figsize=FIG_SIZE)

# Scale bubble sizes for visual clarity (200-800 pixel range)
sizes = priority['segment_size'].values.astype(float)
size_min, size_max = sizes.min(), sizes.max()
# Handle edge case where all segments are the same size
if size_max > size_min:
    bubble_sizes = 200 + 600 * (sizes - size_min) / (size_max - size_min)
else:
    bubble_sizes = np.full_like(sizes, 400.0)

colors = [PALETTE_SEGMENT[s] for s in priority['segment']]

# Plot scatter points first so axes scale to the actual data
for i, (_, p_row) in enumerate(priority.iterrows()):
    ax.scatter(
        p_row['cost_score'], p_row['est_additional_completions'],
        s=bubble_sizes[i], color=colors[i],
        edgecolor='white', linewidth=2, zorder=3, alpha=0.85,
    )
    # Label each bubble with recommendation name
    ax.annotate(
        p_row['recommendation'],
        (p_row['cost_score'], p_row['est_additional_completions']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 12), textcoords='offset points',
    )
    # Annotate with impact number inside/below bubble
    ax.annotate(
        f"~{p_row['est_additional_completions']:,}",
        (p_row['cost_score'], p_row['est_additional_completions']),
        fontsize=8, ha='center', va='top',
        xytext=(0, -10), textcoords='offset points', color='#555555',
    )

# Add subtle quadrant shading AFTER plotting so ylim reflects actual data
ax.axhspan(
    ymin=0, ymax=ax.get_ylim()[1],
    xmin=0, xmax=0.33, alpha=0.04, color='green',
)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Low', 'Medium', 'Medium-High'])
ax.set_xlabel('Implementation Cost')
ax.set_ylabel('Estimated Additional Completions\n(middle scenario)')
ax.set_title(
    'Priority Matrix — Impact vs Cost\n'
    '(bubble size = segment size)'
)
ax.set_xlim(0.4, 3.6)
# Ensure y-axis starts at 0 for honest visual comparison
ax.set_ylim(bottom=0)
sns.despine()
fig.tight_layout()
save_fig(fig, '07_priority_matrix')
plt.show()

> **Leggere la matrice:** L'intervento ideale si trova nell'angolo in alto a sinistra — alto impatto, basso costo. L'Attivazione degli Studenti Ghost è il chiaro "quick win": mira al segmento più grande con il tasso di non-completamento più alto e richiede solo automazione email. Il Checkpoint della Prima Valutazione offre un impatto solido a costo moderato. La Campagna di Re-engagement alla Settimana 3 ha la complessità di implementazione più alta ma mira a una popolazione distinta (non sovrapposta con i ghost), rendendola un'aggiunta preziosa.
>
> **Classifica delle priorità:** (1) Attivazione Ghost — iniziare da qui, (2) Checkpoint Valutazione — costruire dopo, (3) Campagna Re-engagement — investire quando l'infrastruttura è pronta.

## 8. Roadmap di implementazione

Un rollout per fasi minimizza il rischio e permette di validare ogni intervento prima di investire nel successivo.

### Fase 1: Settimane 1–2 — Attivazione degli studenti ghost (Quick Win)

| Step | Azione | Responsabile |
|------|--------|-------------|
| 1.1 | Definire il template email: messaggio di benvenuto + link "primo passo" a una risorsa | Team contenuti |
| 1.2 | Configurare il trigger al giorno 3: lo studente ha zero attività VLE dall'iscrizione | Engineering |
| 1.3 | Configurare il follow-up al giorno 7: ancora inattivo dopo la prima email | Engineering |
| 1.4 | A/B test: 50% riceve le email, 50% gruppo di controllo | Team dati |
| 1.5 | Misurare: tasso di attivazione (% di ghost che accedono al VLE entro 14 giorni) | Team dati |

**Metrica di successo:** ≥15% degli studenti ghost targetizzati accede al VLE entro 7 giorni dalla ricezione dell'email.

### Fase 2: Settimane 3–4 — Checkpoint della prima valutazione

| Step | Azione | Responsabile |
|------|--------|-------------|
| 2.1 | Integrare il calendario delle valutazioni con il sistema di notifiche | Engineering |
| 2.2 | Definire il template del promemoria: anteprima della valutazione + stima del tempo | Team contenuti |
| 2.3 | Configurare il trigger: 3 giorni prima della prima scadenza, lo studente non ha consegnato | Engineering |
| 2.4 | A/B test contro il gruppo di controllo | Team dati |
| 2.5 | Misurare: incremento del tasso di consegna e tasso di completamento a valle | Team dati |

**Metrica di successo:** ≥10% di incremento nel tasso di consegna della prima valutazione tra gli studenti targetizzati.

### Fase 3: Settimane 5–8 — Campagna di re-engagement alla settimana 3

| Step | Azione | Responsabile |
|------|--------|-------------|
| 3.1 | Costruire la pipeline di tracciamento dell'attività in tempo reale (flag di attività giornaliera) | Engineering |
| 3.2 | Implementare la generazione di messaggi personalizzati (riepilogo progressi, statistiche peer) | Engineering + Contenuti |
| 3.3 | Configurare il trigger: 3 giorni consecutivi di inattività dopo l'engagement iniziale | Engineering |
| 3.4 | A/B test con messaggi di re-engagement personalizzati vs generici | Team dati |
| 3.5 | Misurare: tasso di re-engagement e incremento del tasso di completamento | Team dati |

**Metrica di successo:** ≥10% dei disimpegnati targetizzati torna all'attività VLE entro 7 giorni.

### Principio trasversale

Tutti e tre gli interventi mirano al **comportamento, non ai dati demografici** — coerente con il risultato di BQ3 che i segnali comportamentali sono predittori più forti. Questo è sia più efficace (mirare alla variabile azionabile) sia più etico (evitare la profilazione demografica).

## 9. Limitazioni e avvertenze

Queste raccomandazioni sono proiezioni basate sulle evidenze, non risultati garantiti. Diverse limitazioni devono essere riconosciute:

### Assunzioni sulla stima dell'impatto

- **I tassi di conversione sono assunti, non misurati.** Gli scenari di conversione del 10–25% sono plausibili sulla base di benchmark di settore per interventi basati su email nell'istruzione, ma non sono stati validati con i dati OULAD (non esistono dati di A/B test).
- **I tassi di completamento target sono conservativi.** Per l'attivazione ghost, assumiamo che gli studenti convertiti raggiungano la media della piattaforma (non il tasso degli studenti attivi). Per il re-engagement, assumiamo un punto intermedio tra i tassi dei disimpegnati e di chi ha mantenuto l'attività. I risultati effettivi potrebbero essere superiori o inferiori.
- **La sovrapposizione dei segmenti gonfia le somme ingenue degli impatti.** Studenti ghost e non-submitter delle valutazioni si sovrappongono pesantemente. L'impatto combinato di tutti e tre gli interventi è inferiore alla somma delle stime individuali.

### Limitazioni dei dati

- **Solo dati osservazionali.** Tutti gli effect size e le differenze nei tassi di completamento sono associativi, non causali. Uno studente che consegna la prima valutazione potrebbe completare per motivazione, non per la consegna in sé.
- **I pattern storici potrebbero non reggere.** OULAD copre le coorti 2013–2014. Il comportamento degli studenti, le funzionalità delle piattaforme e il panorama dell'apprendimento online sono cambiati significativamente da allora.
- **Nessun dato sui costi.** Le stime dei costi di implementazione (Basso/Medio/Medio-Alto) sono qualitative. L'effort ingegneristico effettivo dipende dall'infrastruttura esistente della piattaforma.

### Considerazioni etiche

- Tutti gli interventi mirano al comportamento, non ai dati demografici — nessuno studente viene profilato per genere, età, disabilità o status socioeconomico.
- Gli interventi automatizzati dovrebbero includere un meccanismo di opt-out per rispettare l'autonomia dello studente.
- Gli "studenti ghost" potrebbero avere ragioni valide per non interagire (circostanze cambiate, iscrizione per errore). L'intervento dovrebbe informare, non pressare.

## 10. Conclusioni chiave

### Risposta a BQ5: Le 3 principali raccomandazioni operative

| Priorità | Intervento | Dimensione segmento | Tasso di non-completamento | Costo | Evidenza chiave |
|----------|-----------|---------------------|---------------------------:|-------|----------------|
| 1 | **Attivazione studenti ghost** | Vedi dimensionamento sopra | Vedi tasso sopra | Basso | BQ2: l'engagement precoce è il predittore più forte |
| 2 | **Checkpoint prima valutazione** | Vedi dimensionamento sopra | Vedi tasso sopra | Medio | BQ2: la consegna della valutazione è un segnale chiave |
| 3 | **Re-engagement settimana 3** | Vedi dimensionamento sopra | Vedi tasso sopra | Medio-Alto | BQ1: cliff di dropout a metà corso nelle settimane 3–4 |

### Sintesi di tutte e 5 le business question

| NB | BQ | Risultato chiave |
|----|-----|-----------------|
| 03 | BQ1 | Il dropout non è uniforme — si concentra in cliff in corrispondenza di specifici milestone del corso. Il ritiro pre-corso è una frazione significativa. |
| 04 | BQ2 | I segnali comportamentali precoci (giorni attivi, click, consegna delle valutazioni) predicono il dropout con effect size elevati. I primi 28 giorni sono la finestra critica. |
| 05 | BQ3 | Il comportamento predice l'esito 2–5× più fortemente dei dati demografici. Gli interventi dovrebbero mirare a cosa *fanno* gli studenti, non a chi *sono*. |
| 06 | BQ4 | I tassi di completamento variano sostanzialmente tra i corsi. Le feature di progettazione del corso (densità delle valutazioni, diversità delle risorse) mostrano associazioni suggestive con la retention. |
| 07 | BQ5 | Tre raccomandazioni operative — attivazione ghost, checkpoint valutazione, campagna di re-engagement — mirano ai più grandi segmenti a rischio con impatto stimato sui dati. |

### La pipeline analitica è completa

Questo notebook conclude il lavoro analitico. Tutti i risultati saranno consolidati in `REPORT.md` per la comunicazione agli stakeholder. Il prossimo passo è il deployment: A/B testing di ogni intervento, misurazione dei tassi di conversione effettivi e iterazione.

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, eseguire prima `python -m run_pipeline`, poi eseguire tutte le celle in ordine.

> **Dall'analisi all'azione:** Questo progetto è iniziato con un dataset e cinque domande. Sette notebook dopo, abbiamo un quadro completo: quando gli studenti se ne vanno, cosa lo predice, cosa conta di più (il comportamento), come differiscono i corsi e — cosa più importante — cosa può fare un operatore di piattaforma. I tre interventi proposti qui non sono speculativi: sono dimensionati su dati reali, supportati da evidenze statistiche e classificati per fattibilità.
>
> Il gap tra analisi e impatto è un A/B test. Questo è il prossimo passo.